# 04.02 VLA 原理、数据采集理论与数据集探索

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 04.01 章节概述</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">理解 VLA/ACT 原理与遥操采集流程，加载并探索 LeRobot 数据集</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">ACT 原理（理论）→ 遥操采集（理论）→ 安装 LeRobot → 数据集探索（实操）</td></tr>
</table>

## 第一部分：ACT 策略原理（理论）

### 模仿学习的核心思想

**模仿学习（Imitation Learning）**：让机器人通过**观察人类演示**学会完成任务，不需要手动编写控制规则。

```text
人类遥操演示 → 记录(观察, 动作)对 → 训练策略 π(动作 | 观察) → 机器人自主执行
```

### ACT 是什么

**ACT（Action Chunking with Transformers）** 是 LeRobot 推荐初学者首选的策略，源自论文 [Learning Fine-Grained Bimanual Manipulation with Low-cost Hardware](https://arxiv.org/abs/2304.13705)。

<img src="./images/act_architecture.png" width="700">

**为什么 ACT 适合初学者**：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">优势</th><th align="left">说明</th></tr>
<tr><td align="left">轻量级</td><td align="left">仅约 80M 参数，单卡 NPU 即可训练</td></tr>
<tr><td align="left">训练快</td><td align="left">100k 步在单卡 NPU 上几小时内完成</td></tr>
<tr><td align="left">数据高效</td><td align="left">通常仅需 50 个演示即可达到高成功率</td></tr>
</table>

### ACT 架构三组件

1. **视觉主干（ResNet-18）**：处理多个摄像头的图像，提取视觉特征；
2. **Transformer 编码器**：综合摄像头特征、关节位置、学习的潜在变量 z；
3. **Transformer 解码器**：用交叉注意力生成**连贯的动作序列块**（一次输出 k 个未来动作）。

**关键创新——动作分块（Action Chunking）**：传统策略一次只预测一个动作，ACT 一次预测一整段动作序列（如未来 100 步），让动作更连贯、减少抖动。

---

## 第二部分：遥操数据采集（理论，需机械臂硬件）

> ⚠️ 本部分为理论讲解。遥操采集需要 SO-101 机械臂硬件，**云环境无法执行**。本节只讲原理和流程，实际采集请参考机械臂实验环境。

### 遥操（Teleoperation）原理

SO-101 采用 **Leader-Follower（主从）双臂模式**：
- **Leader 臂**：人手握住并移动，作为"教师"示范动作；
- **Follower 臂**：实时跟随 Leader 臂的动作，同时记录关节角度和摄像头画面。

```text
人移动 Leader 臂 → Follower 臂跟随 → 记录(摄像头图像, 关节角度) → 形成 1 个 episode
```

### 数据采集流程

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">步骤</th><th align="left">操作</th></tr>
<tr><td align="left">1. 组装校准</td><td align="left">组装机械臂，依次将每个关节转到最小/最大位置完成校准</td></tr>
<tr><td align="left">2. 启动遥操</td><td align="left">用 <code>lerobot-teleop</code> 命令启动 Leader-Follower 模式</td></tr>
<tr><td align="left">3. 录制演示</td><td align="left">人握 Leader 臂完成一次抓取任务，系统自动记录为一个 episode</td></tr>
<tr><td align="left">4. 重复采集</td><td align="left">重复 50-100 次得到足够数据（本课程数据集有 100 episodes）</td></tr>
</table>

### LeRobot 数据集格式

LeRobot v3 数据集结构（本课程数据集 `data_final/` 就是这个格式）：

```text
data_final/
├── data/
│   └── chunk-000/
│       └── file-000.parquet      # 动作与关节状态（表格数据）
├── videos/
│   ├── observation.images.front/  # 前置摄像头视频
│   └── observation.images.wrist/  # 腕部摄像头视频
└── meta/
    ├── info.json                  # 数据集元信息（总episodes、特征定义）
    ├── stats.json                 # 统计信息（均值/标准差，用于归一化）
    └── tasks.parquet              # 任务描述
```

---

## 第三部分：安装 LeRobot 与数据集探索（实操）

### 安装 LeRobot

本章在 CANNLab 选择 **Python 3.11.4 (CANN)** 内核（与第 3 章一致）下运行，使用昇腾 NPU 训练：

In [ ]:
# ===== 安装 LeRobot（首次运行约 5-10 分钟）=====
# CANNLab 环境：选择 cann_py311 内核（Python 3.11.4，与第3章一致），无需另建环境

# 安装 ffmpeg（视频解码需要，Colab 已预装）
# !apt-get install -y ffmpeg    # Colab/Linux
# !brew install ffmpeg           # macOS

# 安装 LeRobot（从 PyPI）
!pip install lerobot -q

# 验证安装并自动检测计算设备（npu → cuda → cpu）
import torch
print(f"PyTorch 版本: {torch.__version__}")

device = None
device_info = ""
# 1) 昇腾 NPU（CANNLab 首选）
try:
    import torch_npu  # noqa: F401  若环境未装 torch_npu，会抛 ImportError
    if torch.npu.is_available():
        device = "npu"
        device_info = f"昇腾 NPU: {torch.npu.get_device_name(0)}"
except ImportError:
    pass
# 2) NVIDIA GPU（本地 / Colab）
if device is None and torch.cuda.is_available():
    device = "cuda"
    device_info = (f"NVIDIA GPU: {torch.cuda.get_device_name(0)}, "
                   f"显存 {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
# 3) CPU 兜底
if device is None:
    device = "cpu"
    device_info = "CPU（无加速卡，仅推荐用于 smoke 测试或推理，训练会很慢）"

print(f"选定设备: {device}  ({device_info})")
print("💡 04.03 训练 / 04.04 推理 notebook 各自也会做同样的设备检测。若 NPU 检测失败，请确认已装 torch_npu。")


### 加载并探索数据集

数据集较大（约 449MB，含视频），无法随课程仓库直接分发。运行下方 cell 会**自动检测并下载**到 `./src/data_final/`，首次下载约 3-10 分钟，后续运行自动跳过。

In [ ]:
# ===== 数据集自动下载（首次运行约 3-10 分钟）=====
# SO-101 方块抓取数据集（449MB）托管在 ModelScope，自动检测本地是否已有
import os

DATA_DIR = './src/data_final'
DATASET_MS_ID = 'Kumako/so101_block_vla'  # ModelScope 数据集 ID
# 完整数据集约 449MB；低于此阈值认为不完整（可能只有 meta 小文件，缺大 mp4）
MIN_COMPLETE_SIZE_MB = 400


def get_dir_size_mb(path):
    """计算目录总大小（MB）"""
    total = 0
    for root, dirs, files in os.walk(path):
        for fname in files:
            fpath = os.path.join(root, fname)
            try:
                total += os.path.getsize(fpath)
            except OSError:
                pass
    return total / (1024 ** 2)


# 检测本地数据集是否完整（看 info.json + 总大小）
has_info = os.path.exists(os.path.join(DATA_DIR, 'meta', 'info.json'))
local_size_mb = get_dir_size_mb(DATA_DIR) if has_info else 0
is_complete = has_info and local_size_mb >= MIN_COMPLETE_SIZE_MB

if is_complete:
    print(f"✅ 本地已有完整数据集: {DATA_DIR}")
    print(f"   数据集大小: {local_size_mb:.1f} MB，跳过下载")
else:
    if has_info:
        print(f"⚠️ 本地数据集不完整（{local_size_mb:.1f} MB < {MIN_COMPLETE_SIZE_MB} MB）")
        print(f"   缺少视频文件等大文件，将重新下载完整数据集")
    else:
        print(f"⏳ 本地未检测到数据集")
    print(f"   开始从 ModelScope 下载...")
    print(f"   数据集 ID: {DATASET_MS_ID}")
    print(f"   约 449MB，首次下载需 3-10 分钟，请耐心等待")
    print("-" * 60)
    try:
        import modelscope
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', 'modelscope',
                             '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple', '-q'])
    from modelscope.hub.snapshot_download import snapshot_download
    # repo_type='dataset' 指明下载数据集仓库
    snapshot_download(DATASET_MS_ID, repo_type='dataset', local_dir=DATA_DIR)
    print("-" * 60)
    new_size = get_dir_size_mb(DATA_DIR)
    print(f"✅ 下载完成！数据集已保存到 {DATA_DIR}")
    print(f"   当前大小: {new_size:.1f} MB")

# 列出数据集结构
print(f"\n数据集目录结构:")
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth <= 1:
        indent = '  ' * depth
        dirname = os.path.basename(root) or DATA_DIR
        # 统计该目录下文件总大小
        dir_size = sum(os.path.getsize(os.path.join(root, f)) for f in files if os.path.isfile(os.path.join(root, f)))
        size_str = f"({dir_size/1024**2:.1f} MB)" if dir_size > 1024*1024 else ""
        file_count = len(files)
        print(f"{indent}{dirname}/  [{file_count}个文件] {size_str}")

In [ ]:
# ===== 加载 LeRobot 数据集 =====
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 指定本地数据集路径（请确保 data_final/ 目录在当前路径下）
dataset = LeRobotDataset(
    repo_id="local/so101_block",
    root="./src/data_final",
    video_backend="pyav"  # NPU环境用pyav解码（torchcodec不兼容）
)

print(f"数据集规模:")
print(f"  总 episodes: {dataset.meta.total_episodes}")
print(f"  总 frames: {dataset.meta.total_frames}")
print(f"  FPS: {dataset.meta.fps}")
print(f"  特征: {list(dataset.meta.features.keys())}")


In [ ]:
# ===== 查看单个样本的结构 =====
# 数据集每条样本包含：观察图像、关节状态、动作
sample = dataset[0]
print("样本结构:")
for key, val in sample.items():
    if hasattr(val, 'shape'):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
    else:
        print(f"  {key}: {type(val).__name__}")


In [ ]:
# ===== 可视化动作分布（6 维动作的统计）=====
# 优化：直接从 meta/stats.json 读全量统计（68146帧），不用逐帧采样
# （逐帧采样会触发视频解码，1000次要几分钟；读 stats.json 毫秒级）
import matplotlib.pyplot as plt
import numpy as np
import json as _json

stats_path = './src/data_final/meta/stats.json'
with open(stats_path) as f:
    stats = _json.load(f)

action_stats = stats['action']  # 含 min/max/mean/std/分位数
action_names = ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, (ax, name) in enumerate(zip(axes.flat, action_names)):
    # 用 min/max 画范围，mean/std 标注
    vmin, vmax = action_stats['min'][i], action_stats['max'][i]
    vmean, vstd = action_stats['mean'][i], action_stats['std'][i]
    # 画一个表示范围的柱状图（min 到 max）
    ax.barh(['range'], [vmax - vmin], left=vmin, color='steelblue', alpha=0.5, height=0.5)
    # 标注 mean（红点）和 ±1σ 范围（绿色横线）
    ax.plot(vmean, ['range'], 'ro', markersize=10, label=f'mean={vmean:.2f}')
    ax.plot([vmean - vstd, vmean + vstd], ['range', 'range'], 'g-', linewidth=3, label=f'±1σ (std={vstd:.2f})')
    ax.set_title(f'action[{i}]: {name}')
    ax.set_xlabel('value (rad)')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_ylim(-0.5, 0.5)
plt.suptitle(f'6-Dim Action Stats Distribution ({action_stats["count"][0]:,} frames, from stats.json)', fontsize=13)
plt.tight_layout()
plt.show()
print("💡 动作分布反映机械臂各关节的活动范围（min~max）和典型值（mean±std）。")
print("   数据已归一化（弧度），mean 接近 0 表示关节常处于中间位置。")


> 💡 **关于数据集规模**：完整数据集 100 episodes / 449MB。如果云存储紧张，可用子集训练：
>
> ```python
> # 只用前 20 episodes（约 90MB）快速测试流程
> dataset = LeRobotDataset(repo_id="local/so101_block", root="./src/data_final", episodes=[0, 1, 2, ..., 19])
> ```
>
> LeRobot 训练对数据量需求不高，ACT 策略 50 episodes 起就能达到不错效果。

---

## 本节练习

**练习 1（选择）**：ACT 策略的"动作分块（Action Chunking）"是什么意思？
- A. 把动作切成小块分别训练
- B. 一次预测一整段未来动作序列，让动作更连贯
- C. 把数据集分成多个块
- D. 减少动作维度

**练习 2（填空）**：SO-101 遥操采用 ______ 模式，人移动 ______ 臂作为教师示范，______ 臂跟随并记录数据。

**练习 3（简答）**：为什么 ACT 适合初学者？至少列出 2 个原因。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/04.02_theory_data/answers.txt
